# ONS Monthly Mortality Counterfactual

This notebook reads England and Wales monthly all-cause deaths from the local MySQL export, fits the pre-2020 counterfactual model, and saves the reference-style chart.


In [ ]:
from pathlib import Path

import pandas as pd

from ons_mortality.counterfactual import CounterfactualConfig, fit_counterfactual, plot_counterfactual
from ons_mortality.database import create_mysql_engine, read_national_monthly_deaths


## Load data

The notebook first looks for a local CSV export. If it is not present, it reads the national series from MySQL.


In [ ]:
input_csv = Path('../data/processed/england_wales_monthly_deaths.csv')

if input_csv.exists():
    deaths_df = pd.read_csv(input_csv)
else:
    engine = create_mysql_engine()
    deaths_df = read_national_monthly_deaths(engine)

deaths_df.head()


## Fit counterfactual

The model uses the monthly observations before March 2020 and projects the no-pandemic trajectory forward.


In [ ]:
config = CounterfactualConfig(
    pandemic_onset='2020-03-01',
    n_posterior_samples=8_000,
    random_seed=42,
)

result = fit_counterfactual(deaths_df, config=config)
result.tail()


## Plot

The chart is saved under `figures/`.


In [ ]:
output_path = Path('../figures/england_wales_counterfactual.png')
plot_counterfactual(result, output_path=output_path, config=config)
output_path
